# recipes3.js 데이터를 불러와서, 카테고리와 재료 정보를 결합해 레시피 간의 유사도를 계산

In [3]:
import pandas as pd # 데이터를 표(DataFrame) 형태로 관리하고 조작하기 위한 핵심 라이브러리
import json # 크롤링한 JSON 형식의 데이터 파일을 파이썬 객체로 불러오기 위해 사용
from sklearn.feature_extraction.text import TfidfVectorizer # 텍스트를 숫자로 된 벡터로 변환(TF-IDF 방식 적용)
from sklearn.metrics.pairwise import cosine_similarity # 숫자로 변환된 벡터들 사이의 유사도를 계산(코사인 각도 사용)

In [4]:
# [데이터 로드 및 가공]
# 프로젝트 구조상 JS 모듈 형식으로 저장된 데이터를 파이썬에서 읽을 수 있게 처리
with open('/Users/gangseongbin/login-project/src/data/collection/recipes3.js', 'r', encoding='utf-8') as f:
    # JS의 'export const' 구문을 제거하여 순수 JSON 데이터만 추출
    content = f.read().replace('export const recipes3 = ', '').rstrip(';')
    data = json.loads(content)
    df = pd.DataFrame(data) # 분석을 위해 데이터를 표 형태로 변환

데이터 정제 및 조합: category와 ingredients에 있는 재료명만 뽑아 하나의 텍스트 덩어리로 만듭니다. (이것이 벡터의 원재료가 됩니다.)

In [5]:
# [데이터 특징 추출]
# 카테고리와 재료 정보를 하나의 텍스트로 결합하여 '레시피의 특징'을 정의
def preprocess_recipe(row):
    # 재료 리스트(amount)에서 단위(g, T 등)를 제외하고 재료명만 추출하여 공백으로 연결
    ingredients = " ".join([item['amount'].split()[0] for item in row['ingredients']])
    return f"{row['category']} {ingredients}" # 예: "한식 마늘 진간장 올리브유..."

In [6]:
df['combined_features'] = df.apply(preprocess_recipe, axis=1)

# [벡터화 및 학습]
# TfidfVectorizer: 각 단어의 빈도와 전체 문서에서의 희소성을 고려해 가중치 부여 (중요한 재료 단어일수록 가중치 상승)
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df['combined_features']) # 텍스트를 고차원 숫자 벡터로 변환

# [유사도 계산]
# 모든 레시피와 모든 레시피 사이의 유사도를 행렬 형태로 계산
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

TF-IDF 벡터화: "한식", "간장", "마늘"과 같은 단어들에 가중치를 주어 숫자로 이루어진 리스트(벡터)로 변환합니다.

In [8]:
# [추천 실행]
# 특정 레시피와 가장 유사한 벡터를 가진 다른 레시피들을 추천
def get_recommendations(title, cosine_sim=cosine_sim):
    # 입력한 제목이 포함된 레시피의 인덱스 검색
    idx = df[df['title'].str.contains(title)].index[0]
    
    # 해당 인덱스의 유사도 점수들을 정렬하여 상위 레시피 추출
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:6] # 본인 제외 유사도 높은 5개
    
    recipe_indices = [i[0] for i in sim_scores]
    return df['title'].iloc[recipe_indices]

In [26]:
def get_recommendations_by_ingredients(user_ingredients):
    # 1. 재료 필터링 (정확한 매칭)
    def has_ingredients(ingredients_list, query_ingredients):
        # 데이터의 모든 'amount' 값을 하나의 문자열로 합침
        full_ingredients_text = " ".join([item.get('amount', '') for item in ingredients_list])
        # 사용자가 입력한 재료들이 그 문자열 안에 있는지 확인
        return any(query in full_ingredients_text for query in query_ingredients)

    # 2. 리스트 형태의 입력 처리
    if isinstance(user_ingredients, str):
        user_ingredients = [user_ingredients]

    # 3. 데이터프레임 필터링
    mask = df['ingredients'].apply(lambda x: has_ingredients(x, user_ingredients))
    filtered_df = df[mask]
    
    # 결과 확인용 로그 (데이터가 진짜 있는지 확인)
    print(f"검색된 레시피 수: {len(filtered_df)}")
    
    if filtered_df.empty:
        return []

    # 4. 필터링된 데이터셋으로 유사도 계산
    filtered_indices = filtered_df.index
    sub_matrix = tfidf_matrix[filtered_indices]
    
    query_vec = tfidf.transform([" ".join(user_ingredients)])
    sim_scores = cosine_similarity(query_vec, sub_matrix).flatten()
    
    top_relative_indices = sim_scores.argsort()[-5:][::-1]
    
    return filtered_df.iloc[top_relative_indices]['title'].tolist()

In [25]:
# 실행 예시
# 데이터 구조 확인용
first_recipe = df.iloc[0]['ingredients']
print(f"데이터 타입: {type(first_recipe)}")
print(f"내용물 예시: {first_recipe}")

데이터 타입: <class 'list'>
내용물 예시: [{'ingredientId': 1, 'amount': '마늘 1알'}, {'ingredientId': 2, 'amount': '진간장 2T'}, {'ingredientId': 3, 'amount': '올리브유 2T'}, {'ingredientId': 4, 'amount': '참기름 1T'}, {'ingredientId': 5, 'amount': '설탕 1T'}, {'ingredientId': 6, 'amount': '식초 1T'}, {'ingredientId': 7, 'amount': '겨자 약간'}, {'ingredientId': 8, 'amount': '후춧가루 약간'}]


In [27]:
get_recommendations_by_ingredients("고기")

검색된 레시피 수: 1


['한식의 대표주자! 남녀노소 누구나 좋아하는 맛있는 소불고기만들기']